In [ ]:
# ═══════════════════════════════════════════════
# STABLE MONEY AI AVATAR — GPU + LIP SYNC
# Run this single cell on Google Colab (GPU runtime)
# ═══════════════════════════════════════════════

import subprocess, os, sys, time

# Check GPU
import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
print(f"GPU: {gpu_name}")
if not torch.cuda.is_available():
    print("WARNING: No GPU! Lip sync will be disabled. Go to Runtime -> Change runtime type -> GPU")

# Install dependencies
print("Installing dependencies...")
os.system("pip install -q fastapi 'uvicorn[standard]' python-multipart aiofiles")
os.system("pip install -q edge-tts groq")
os.system("pip install -q pyngrok nest_asyncio")
os.system("pip install -q face-detection opencv-python-headless")
os.system("apt-get -qq install -y ffmpeg > /dev/null 2>&1")
print("Dependencies installed")

# Clone Wav2Lip
if not os.path.exists('/content/Wav2Lip'):
    print("Cloning Wav2Lip...")
    os.system("git clone -q https://github.com/Rudrabha/Wav2Lip.git /content/Wav2Lip")
    print("Wav2Lip cloned")

# Download Wav2Lip checkpoint
os.makedirs('/content/checkpoints', exist_ok=True)
if not os.path.exists('/content/checkpoints/wav2lip_gan.pth'):
    print("Downloading Wav2Lip checkpoint (~400MB)...")
    os.system("wget -q 'https://iiitaphyd-my.sharepoint.com/personal/radrabha_m_research_iiit_ac_in/_layouts/15/download.aspx?share=EdjI7bZlgApMqsVoEUUXpLsBxqXbn5z8VTmoxp55YNDcIA' -O /content/checkpoints/wav2lip_gan.pth")
    # Alternative URL if above doesn't work:
    # os.system("gdown 'https://drive.google.com/uc?id=1qjFOkT4xCHLLIU8E2ERsgqBMOlKWDd-k' -O /content/checkpoints/wav2lip_gan.pth")
    print("Checkpoint downloaded")

# Download avatar photo
if not os.path.exists('/content/avatar.jpg'):
    print("Downloading avatar photo...")
    os.system("wget -q 'https://randomuser.me/api/portraits/men/32.jpg' -O /content/avatar.jpg")
    print("Avatar downloaded")

# Set environment variables
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

# ── Write server.py ──
print("Writing server.py...")
server_code = r'''"""
Stable Money Avatar Agent — Self-Hosted Backend
Replaces: ElevenLabs (Coqui XTTS v2) + HeyGen (Wav2Lip)
Run: uvicorn server:app --host 0.0.0.0 --port 8000
"""

import asyncio, base64, io, os, sys, tempfile, time, uuid
from pathlib import Path
from typing import Optional

import cv2
import numpy as np
import torch
from fastapi import FastAPI, WebSocket, WebSocketDisconnect, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from fastapi.staticfiles import StaticFiles
import soundfile as sf

# ── Coqui TTS ──
try:
    from TTS.api import TTS
    COQUI_AVAILABLE = True
except ImportError:
    COQUI_AVAILABLE = False

# ── Wav2Lip (optional — requires GPU server) ──
WAV2LIP_AVAILABLE = False
try:
    sys.path.insert(0, "./Wav2Lip")
    import audio as wav2lip_audio
    from models import Wav2Lip as Wav2LipModel
    import face_detection
    WAV2LIP_AVAILABLE = True
    print("[server] Wav2Lip available")
except ImportError:
    print("[server] Wav2Lip not available — avatar disabled, voice-only mode")

app = FastAPI(title="Stable Money Avatar Server")

import asyncio

app.add_middleware(CORSMiddleware,
    allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

# ── Config ──
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
SAMPLE_RATE  = 24000
AVATAR_IMG   = "./avatar.jpg"          # uploaded face image
WAV2LIP_CKPT = "./checkpoints/wav2lip_gan.pth"
VOICE_SAMPLE = "./voice_sample.wav"    # 6s sample for voice cloning

print(f"[server] Using device: {DEVICE}")

# ════════════════════════════════════════
# 1. LOAD MODELS AT STARTUP
# ════════════════════════════════════════

class Models:
    tts = None
    wav2lip = None
    face_frames = None   # pre-processed face frames
    detector = None

models = Models()

@app.on_event("startup")
async def load_models():
    if COQUI_AVAILABLE:
        print("[startup] Loading Coqui XTTS v2...")

        _torch_load = torch.load
        def _patched_torch_load(*args, **kwargs):
            if "weights_only" not in kwargs:
                kwargs["weights_only"] = False
            return _torch_load(*args, **kwargs)
        torch.load = _patched_torch_load

        models.tts = TTS("tts_models/en/ljspeech/tacotron2-DDC").to(DEVICE)
        print("[startup] XTTS v2 loaded")
    else:
        print("[startup] Coqui TTS not available, using edge-tts only")

    if Path(WAV2LIP_CKPT).exists():
        print("[startup] Loading Wav2Lip...")
        ckpt = torch.load(WAV2LIP_CKPT, map_location=DEVICE)
        s = ckpt["state_dict"]
        # strip 'module.' prefix if present
        s = {k.replace("module.", ""): v for k, v in s.items()}
        models.wav2lip = Wav2LipModel()
        models.wav2lip.load_state_dict(s)
        models.wav2lip = models.wav2lip.to(DEVICE).eval()
        print("[startup] Wav2Lip loaded")

        models.detector = face_detection.FaceAlignment(
            face_detection.LandmarksType._2D,
            flip_input=False, device=DEVICE)

        if Path(AVATAR_IMG).exists():
            await preprocess_face(AVATAR_IMG)
    else:
        print("[startup] Wav2Lip checkpoint not found — avatar disabled")

    print("[startup] All models ready")


# ════════════════════════════════════════
# 2. TTS — YOUR OWN ELEVENLABS
# ════════════════════════════════════════

ACRONYMS = {
    "DICGC": "D I C G C",
    "NBFC": "N B F C",
    "NBFCs": "N B F Cs",
    "SEBI": "S E B I",
    "RBI": "R B I",
    # FD/FDs intentionally excluded — LLM already writes "Fixed Deposits (FDs)"
    # so replacing "FDs" would create "Fixed Deposits (fixed deposits)" double-read
    "SFB": "small finance bank",
    "SFBs": "small finance banks",
}

def preprocess_tts(text: str) -> str:
    import re
    # Step 1: "ACRONYM (Full Form)" -> keep only Full Form, drop the redundant acronym
    # e.g. "RBI (Reserve Bank of India)" -> "Reserve Bank of India"
    # e.g. "NBFCs (Non-Banking Financial Companies)" -> "Non-Banking Financial Companies"
    text = re.sub(r'\b[A-Z]{2,7}s?\s*\(([^)]+)\)', r'\1', text)
    # Step 2: expand any remaining standalone acronyms
    for acronym, expansion in ACRONYMS.items():
        text = re.sub(r'\b' + re.escape(acronym) + r'\b', expansion, text)
    return text

def _detect_hindi(text: str) -> bool:
    """Return True if text contains Devanagari characters (Hindi)."""
    import re
    return bool(re.search(r'[\u0900-\u097F]', text))

# ── Knowledge Base ──
_KB_PATH = Path("./knowledge_base.txt")
def _load_kb() -> str:
    if _KB_PATH.exists():
        return _KB_PATH.read_text(encoding="utf-8")
    return ""

KNOWLEDGE_BASE = _load_kb()

async def synthesize_speech(text: str) -> bytes:
    """Convert text -> MP3 bytes using edge-tts with Indian voice."""
    import edge_tts
    text = preprocess_tts(text)
    voice = "hi-IN-MadhurNeural" if _detect_hindi(text) else "en-IN-PrabhatNeural"
    communicate = edge_tts.Communicate(text, voice=voice, rate="+5%", pitch="+0Hz")
    buf = io.BytesIO()
    async for chunk in communicate.stream():
        if chunk["type"] == "audio":
            buf.write(chunk["data"])
    buf.seek(0)
    return buf.read()


# ════════════════════════════════════════
# 3. AVATAR — YOUR OWN HEYGEN
# ════════════════════════════════════════

IMG_SIZE = 96   # Wav2Lip input size

async def preprocess_face(img_path: str):
    """Pre-process face image for Wav2Lip — run once"""
    img = cv2.imread(img_path)
    img = cv2.resize(img, (640, 480))
    # detect face bbox
    preds = models.detector.get_detections_for_batch(
        np.array([img[:, :, ::-1]])
    )
    if not preds or preds[0] is None:
        print("[avatar] No face detected in image!")
        return
    # store as repeated frames (Wav2Lip needs video-like input)
    models.face_frames = [img] * 25  # 25 still frames
    print(f"[avatar] Face pre-processed")


def generate_lipsync_video(audio_bytes: bytes) -> Optional[bytes]:
    """
    Audio bytes -> lip-synced MP4 bytes
    Core Wav2Lip inference pipeline
    """
    if models.wav2lip is None or models.face_frames is None:
        return None

    # Write audio to temp file
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
        f.write(audio_bytes)
        audio_path = f.name

    out_path = f"/tmp/{uuid.uuid4().hex}.mp4"

    try:
        # Load & process mel spectrogram
        wav = wav2lip_audio.load_wav(audio_path, 16000)
        mel = wav2lip_audio.melspectrogram(wav)
        mel_chunks = []
        mel_step  = 16
        mel_idx   = 0
        fps       = 25
        mel_per_frame = mel.shape[1] / (len(wav) / 16000 * fps)

        for _ in models.face_frames:
            s = int(mel_idx)
            e = s + mel_step
            mel_chunks.append(mel[:, s:e])
            mel_idx += mel_per_frame

        # Extend face frames to match mel
        face_frames_extended = (models.face_frames * (
            (len(mel_chunks) // len(models.face_frames)) + 1
        ))[:len(mel_chunks)]

        # Wav2Lip inference in batches
        BATCH = 128
        gen_frames = []

        for i in range(0, len(mel_chunks), BATCH):
            img_batch  = np.array(face_frames_extended[i:i+BATCH])
            mel_batch  = np.array(mel_chunks[i:i+BATCH])

            # Prepare lower-half mask (Wav2Lip only animates mouth region)
            img_masked = img_batch.copy()
            img_masked[:, img_batch.shape[1]//2:] = 0

            img_batch_t = torch.FloatTensor(
                np.transpose(img_batch, (0,3,1,2))).to(DEVICE) / 255.
            img_masked_t = torch.FloatTensor(
                np.transpose(img_masked, (0,3,1,2))).to(DEVICE) / 255.
            mel_batch_t = torch.FloatTensor(
                np.transpose(mel_batch, (0,3,1,2))).unsqueeze(1).to(DEVICE)

            with torch.no_grad():
                pred = models.wav2lip(mel_batch_t, img_masked_t)

            pred = (pred.permute(0,2,3,1).cpu().numpy() * 255).astype(np.uint8)

            for p, orig in zip(pred, img_batch):
                # Blend generated mouth region back into original frame
                h, w = orig.shape[:2]
                p_resized = cv2.resize(p, (w, h))
                combined  = orig.copy()
                combined[h//2:] = p_resized[h//2:]
                gen_frames.append(combined)

        # Write to MP4
        h, w = gen_frames[0].shape[:2]
        writer = cv2.VideoWriter(
            out_path,
            cv2.VideoWriter_fourcc(*"mp4v"),
            fps, (w, h)
        )
        for f in gen_frames:
            writer.write(f)
        writer.release()

        # Mux audio into video using ffmpeg
        muxed = out_path.replace(".mp4", "_final.mp4")
        os.system(
            f"ffmpeg -y -i {out_path} -i {audio_path} "
            f"-c:v copy -c:a aac -shortest {muxed} -loglevel quiet"
        )

        with open(muxed, "rb") as vf:
            return vf.read()

    finally:
        for p in [audio_path, out_path, out_path.replace(".mp4","_final.mp4")]:
            try: os.remove(p)
            except: pass


# ════════════════════════════════════════
# 4. WEBSOCKET — REAL-TIME PIPELINE
# ════════════════════════════════════════

@app.websocket("/ws/talk")
async def ws_talk(ws: WebSocket):
    """
    Minimal working WebSocket:
    receives text, returns text + audio
    """
    await ws.accept()
    print("[ws] Client connected")

    try:
        while True:
            data = await ws.receive_json()
            msg_type = data.get("type")

            if msg_type == "config":
                print("[ws] config received")
                continue

            if msg_type != "speak":
                continue

            text = data.get("text", "").strip()
            if not text:
                continue

            # Use explicit lang sent by frontend; fallback to Devanagari detection
            lang = data.get("lang") or ("hi" if _detect_hindi(text) else "en")

            if lang == "hi":
                system_msg = f"""You are StableAI, Stable Money's expert financial advisor.

OFFICIAL KNOWLEDGE BASE:
{KNOWLEDGE_BASE}

Strict rules:
- Reply ONLY in Hindi — no English sentences.
- Answer the question directly — no filler like "great question".
- 2-3 short, clear sentences are enough. No long speeches.
- Only share information from the knowledge base — do not invent.
- Never write "RBI (Reserve Bank of India)" style — no acronym + full form together. Use full form only.
- Always complete your answer, never stop mid-sentence.
- If the answer is not in the knowledge base, say: I will confirm with the team and get back to you."""
            else:
                system_msg = f"""You are StableAI, a sharp and friendly male financial advisor at Stable Money.

OFFICIAL KNOWLEDGE BASE (use ONLY this — never invent facts):
{KNOWLEDGE_BASE}

STRICT RULES:
- Answer ONLY in English. Zero Hindi words.
- Get straight to the point. No "Great question!", no fluff, no repeating the question.
- Keep it to 2-3 sentences — clear, specific, and useful.
- Use exact numbers from the knowledge base (e.g. "8% to 9.5%", "Rs 5 lakhs cover").
- NEVER write an acronym and its full form together like "RBI (Reserve Bank of India)" — pick one. Prefer the full form for clarity.
- If the answer isn't in the knowledge base, say: "Let me check the latest details on that and get back to you."
- Always finish your sentence — never trail off mid-answer."""

            try:
                from groq import Groq
                client = Groq(api_key=os.environ.get("GROQ_API_KEY", ""))
                completion = client.chat.completions.create(
                    model="llama-3.1-8b-instant",
                    messages=[
                        {"role": "system", "content": system_msg},
                        {"role": "user", "content": text}
                    ],
                    max_tokens=150,
                    temperature=0.6,
                )
                answer = completion.choices[0].message.content.strip().strip('"') or "Sorry, I could not generate a response."
            except Exception as e:
                answer = f"Sorry, Groq failed: {str(e)}"

            print(f"[ws] answer ready: {answer[:120]}")
            await ws.send_json({"type": "text_chunk", "text": answer})
            await ws.send_json({"type": "status", "msg": "synthesizing"})

            try:
                audio_bytes = await synthesize_speech(answer)
            except Exception as e:
                await ws.send_json({"type": "error", "msg": f"TTS failed: {str(e)}"})
                continue

            # If Wav2Lip available (GPU), generate lip-synced video
            video_bytes = None
            if models.wav2lip is not None and models.face_frames is not None:
                try:
                    await ws.send_json({"type": "status", "msg": "animating"})
                    # edge-tts outputs MP3, Wav2Lip needs WAV — convert
                    wav_path = f"/tmp/{uuid.uuid4().hex}.wav"
                    mp3_path = f"/tmp/{uuid.uuid4().hex}.mp3"
                    with open(mp3_path, "wb") as f:
                        f.write(audio_bytes)
                    os.system(f"ffmpeg -y -i {mp3_path} -ar 16000 -ac 1 {wav_path} -loglevel quiet")
                    with open(wav_path, "rb") as f:
                        wav_bytes = f.read()
                    for p in [mp3_path, wav_path]:
                        try: os.remove(p)
                        except: pass
                    video_bytes = generate_lipsync_video(wav_bytes)
                except Exception as e:
                    print(f"[ws] Wav2Lip failed, falling back to audio: {e}")

            if video_bytes:
                await ws.send_json({
                    "type": "video",
                    "data": base64.b64encode(video_bytes).decode("utf-8"),
                    "format": "mp4"
                })
            else:
                await ws.send_json({
                    "type": "audio",
                    "data": base64.b64encode(audio_bytes).decode("utf-8"),
                    "format": "mp3"
                })

            await ws.send_json({"type": "done"})
            print("[ws] Ready for next question")

    except WebSocketDisconnect:
        print("[ws] Client disconnected")
    except Exception as e:
        print("[ws] Error:", e)
        try:
            await ws.send_json({"type": "error", "msg": str(e)})
        except:
            pass


@app.post("/upload/avatar")
async def upload_avatar(file: UploadFile = File(...)):
    contents = await file.read()
    with open(AVATAR_IMG, "wb") as f:
        f.write(contents)
    return {"status": "ok", "message": "Avatar photo saved"}

@app.post("/upload/voice")
async def upload_voice(file: UploadFile = File(...)):
    contents = await file.read()
    with open(VOICE_SAMPLE, "wb") as f:
        f.write(contents)
    return {"status": "ok", "message": "Voice sample saved — cloning active"}

@app.get("/health")
def health():
    return {
        "status": "ok",
        "device": DEVICE,
        "tts_loaded": models.tts is not None,
        "wav2lip_loaded": models.wav2lip is not None,
        "face_ready": models.face_frames is not None,
        "voice_clone": Path(VOICE_SAMPLE).exists()
    }

# ── Static frontend ──
if Path("./static").exists():
    app.mount("/", StaticFiles(directory="static", html=True), name="static")
'''
with open('/content/server.py', 'w') as f:
    f.write(server_code)
print("server.py written")

# ── Write index.html ──
print("Writing index.html...")
os.makedirs('/content/static', exist_ok=True)
index_html = r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width,initial-scale=1"/>
<meta http-equiv="Cache-Control" content="no-cache, no-store, must-revalidate"/>
<title>StableAI</title>
<style>
*{margin:0;padding:0;box-sizing:border-box;}
body{background:#0a0a0f;color:#fff;font-family:-apple-system,sans-serif;min-height:100vh;display:flex;flex-direction:column;align-items:center;justify-content:center;gap:16px;}
.status-bar{position:fixed;top:16px;left:50%;transform:translateX(-50%);background:rgba(255,255,255,0.08);border:1px solid rgba(255,255,255,0.12);border-radius:20px;padding:6px 16px;font-size:13px;display:flex;align-items:center;gap:8px;}
.dot{width:8px;height:8px;border-radius:50%;background:#666;}
.dot.green{background:#4ade80;}.dot.gold{background:#fbbf24;}.dot.blue{background:#60a5fa;}.dot.red{background:#f87171;}
.settings-btn{position:fixed;top:16px;right:16px;z-index:9999;background:rgba(255,255,255,0.08);border:1px solid rgba(255,255,255,0.12);border-radius:20px;padding:6px 14px;font-size:13px;cursor:pointer;color:#fff;}
.settings-panel{position:fixed;top:56px;right:16px;z-index:9999;background:#1a1a2e;border:1px solid rgba(255,255,255,0.15);border-radius:12px;padding:16px;width:300px;display:none;}
.settings-panel.open{display:block;}
.settings-panel input{width:100%;background:rgba(255,255,255,0.08);border:1px solid rgba(255,255,255,0.2);border-radius:8px;padding:8px;color:#fff;font-size:13px;margin:8px 0;}
.settings-panel button{width:100%;background:#4a90d9;border:none;border-radius:8px;padding:8px;color:#fff;cursor:pointer;}

/* ── Face card — WhatsApp video-call style ── */
.face-card{width:340px;height:440px;background:#0e0e18;border:1px solid rgba(255,255,255,0.06);border-radius:20px;display:flex;flex-direction:column;align-items:center;position:relative;overflow:hidden;transition:box-shadow 0.3s;}
.face-card.speaking{box-shadow:0 0 0 2px rgba(74,144,217,0.7),0 0 40px rgba(74,144,217,0.15);}

/* Thinking state: gentle pulsing glow */
@keyframes thinkPulse{
  0%,100%{box-shadow:0 0 0 2px rgba(251,191,36,0.4),0 0 20px rgba(251,191,36,0.05);}
  50%    {box-shadow:0 0 0 2px rgba(251,191,36,0.7),0 0 40px rgba(251,191,36,0.15);}
}
.face-card.thinking{animation:thinkPulse 1.8s ease-in-out infinite;}

/* Avatar photo — fills the card like a video call */
.avatar-photo{width:100%;height:100%;object-fit:cover;object-position:center top;display:block;}

/* Subtle head sway when idle, breathing when speaking */
@keyframes idleSway{
  0%,100%{transform:scale(1.04) translate(0,0);}
  25%    {transform:scale(1.04) translate(1.5px,-1px);}
  50%    {transform:scale(1.04) translate(-1px,1px);}
  75%    {transform:scale(1.04) translate(1px,0.5px);}
}
@keyframes speakSway{
  0%,100%{transform:scale(1.04) translate(0,0);}
  30%    {transform:scale(1.05) translate(1px,-1.5px);}
  60%    {transform:scale(1.04) translate(-1.5px,0.5px);}
}
.avatar-photo{animation:idleSway 6s ease-in-out infinite;}
.face-card.speaking .avatar-photo{animation:speakSway 2.5s ease-in-out infinite;}

/* Name overlay at bottom like a video call */
.name-overlay{position:absolute;bottom:0;left:0;right:0;padding:12px 16px;background:linear-gradient(transparent,rgba(0,0,0,0.7));display:flex;align-items:center;justify-content:space-between;}
.name-overlay h2{font-size:17px;font-weight:600;}
.name-overlay p{font-size:11px;color:rgba(255,255,255,0.5);}

/* Sound wave visualizer overlaid bottom-right */
.sound-wave{display:none;align-items:flex-end;gap:3px;height:18px;}
.face-card.speaking .sound-wave{display:flex;}
.sound-wave span{width:3px;background:#4ade80;border-radius:2px;animation:wave 0.8s ease-in-out infinite;}
.sound-wave span:nth-child(1){animation-delay:0s;height:6px;}
.sound-wave span:nth-child(2){animation-delay:0.1s;height:12px;}
.sound-wave span:nth-child(3){animation-delay:0.2s;height:18px;}
.sound-wave span:nth-child(4){animation-delay:0.1s;height:12px;}
.sound-wave span:nth-child(5){animation-delay:0s;height:6px;}
@keyframes wave{0%,100%{transform:scaleY(0.4);}50%{transform:scaleY(1);}}

/* ── Response + transcript bubbles (below face card) ── */
.response-bubble{width:100%;max-width:480px;background:rgba(255,255,255,0.05);border:1px solid rgba(255,255,255,0.12);border-radius:16px;padding:12px 16px;font-size:14px;line-height:1.6;display:none;}
.transcript{width:100%;max-width:480px;background:rgba(74,144,217,0.1);border:1px solid rgba(74,144,217,0.25);border-radius:12px;padding:8px 14px;font-size:12px;color:rgba(255,255,255,0.6);display:none;}

/* Thinking dots animation inside response bubble */
@keyframes dotBounce{
  0%,80%,100%{transform:translateY(0);}
  40%{transform:translateY(-8px);}
}
.thinking-dots{display:inline-flex;gap:4px;padding:4px 0;}
.thinking-dots span{width:8px;height:8px;border-radius:50%;background:rgba(255,255,255,0.4);animation:dotBounce 1.2s ease-in-out infinite;}
.thinking-dots span:nth-child(2){animation-delay:0.15s;}
.thinking-dots span:nth-child(3){animation-delay:0.3s;}

.discuss-btn{background:linear-gradient(135deg,#4a90d9,#357abd);border:none;border-radius:24px;padding:14px 52px;font-size:17px;font-weight:600;color:#fff;cursor:pointer;}
.discuss-btn:disabled{opacity:0.5;cursor:default;}
.chips{display:flex;flex-wrap:wrap;gap:8px;justify-content:center;max-width:480px;padding:0 16px;}
.chip{background:rgba(255,255,255,0.07);border:1px solid rgba(255,255,255,0.12);border-radius:20px;padding:8px 14px;font-size:13px;cursor:pointer;}
.chip:hover{background:rgba(255,255,255,0.13);}
/* Video playback in face card */
.avatar-video{position:absolute;top:0;left:0;width:100%;height:100%;object-fit:cover;object-position:center top;z-index:1;display:none;}
.avatar-video.active{display:block;}
</style>
</head>
<body>
<div class="status-bar"><div class="dot" id="dot"></div><span id="statusText">Connecting...</span></div>
<button class="settings-btn" onclick="toggleSettings()">Settings</button>
<div class="settings-panel" id="settingsPanel">
  <div style="font-size:13px;color:rgba(255,255,255,0.5);margin-bottom:4px;">WebSocket URL</div>
  <input type="text" id="serverUrl" value="ws://localhost:8000/ws/talk"/>
  <button onclick="connectServer()">Connect</button>
</div>
<div class="face-card" id="faceCard">
  <img class="avatar-photo" id="avatarImg"
       src="https://randomuser.me/api/portraits/men/32.jpg"
       onerror="this.src='data:image/svg+xml,<svg xmlns=%22http://www.w3.org/2000/svg%22 width=%22340%22 height=%22440%22><rect fill=%22%23223%22 width=%22340%22 height=%22440%22/><text x=%2250%25%22 y=%2250%25%22 text-anchor=%22middle%22 fill=%22%23667%22 font-size=%2260%22 font-family=%22sans-serif%22>AI</text></svg>'"
       alt="StableAI"/>
  <video class="avatar-video" id="avatarVideo" muted playsinline></video>
  <div class="name-overlay">
    <div><h2>StableAI</h2><p>Fixed Income Advisor</p></div>
    <div class="sound-wave"><span></span><span></span><span></span><span></span><span></span></div>
  </div>
</div>
<div class="transcript" id="transcriptBubble"></div>
<div class="response-bubble" id="responseBubble"></div>
<div style="display:flex;gap:10px;align-items:center;">
  <button class="discuss-btn" id="discussBtn" onclick="startDiscuss()">Discuss</button>
  <button id="langBtn" onclick="toggleLang()" style="background:rgba(255,255,255,0.08);border:1px solid rgba(255,255,255,0.2);border-radius:20px;padding:10px 18px;font-size:14px;color:#fff;cursor:pointer;">EN</button>
</div>
<div class="chips">
  <div class="chip" onclick="sendText('What is Stable Money?')">What is Stable Money?</div>
  <div class="chip" onclick="sendText('How safe are FDs?')">How safe are FDs?</div>
  <div class="chip" onclick="sendText('Expected returns?')">Expected returns?</div>
  <div class="chip" onclick="sendText('Tell me about Sukoon')">Tell me about Sukoon</div>
  <div class="chip" onclick="sendText('Better than bank FDs?')">Better than bank FDs?</div>
  <div class="chip" onclick="sendText('Min investment?')">Min investment?</div>
</div>
<script>
const S={ws:null,ready:false,speaking:false,continuous:false,audioCtx:null,
         lang:"en-IN",currentAudio:null,sentCount:0,doneCount:0,hideTimer:null};

function toggleSettings(){document.getElementById("settingsPanel").classList.toggle("open");}
function setStatus(txt,color){document.getElementById("statusText").textContent=txt;document.getElementById("dot").className="dot"+(color?" "+color:"");}

function connectServer(){
  const url=document.getElementById("serverUrl").value.trim();
  document.getElementById("settingsPanel").classList.remove("open");
  setStatus("Connecting...","gold");
  try{
    S.ws=new WebSocket(url);
    S.ws.onopen=()=>{S.ws.send(JSON.stringify({type:"config"}));S.ready=true;setStatus("Ready - Tap Discuss","green");};
    S.ws.onmessage=e=>handleMsg(JSON.parse(e.data));
    S.ws.onerror=()=>{setStatus("Connection failed","red");S.ready=false;};
    S.ws.onclose=()=>{setStatus("Disconnected","red");S.ready=false;};
  }catch(e){setStatus("Error: "+e.message,"red");}
}

/* ── Show thinking animation immediately ── */
function showThinking(){
  const fc=document.getElementById("faceCard");
  fc.classList.add("thinking");
  fc.classList.remove("speaking");
  const b=document.getElementById("responseBubble");
  b.innerHTML='<div class="thinking-dots"><span></span><span></span><span></span></div>';
  b.style.display="block";
}
function hideThinking(){
  document.getElementById("faceCard").classList.remove("thinking");
}

function handleMsg(msg){
  const isStale = S.doneCount < S.sentCount - 1;
  if(isStale){
    if(msg.type==="done"||msg.type==="error") S.doneCount++;
    return;
  }
  if(msg.type==="text_chunk"){
    hideThinking();
    const b=document.getElementById("responseBubble");
    b.textContent=msg.text;b.style.display="block";
  }
  if(msg.type==="video") playVideo(msg.data,msg.format);
  else if(msg.type==="audio") playAudio(msg.data,msg.format);
  if(msg.type==="done"||msg.type==="error"){
    S.doneCount++;
    hideThinking();
    if(msg.type==="error") setStatus("Error","red");
    if(!S.speaking) afterResponse();
  }
}

function playVideo(b64,fmt){
  try{S.recognition&&S.recognition.abort();}catch(e){}
  const bytes=Uint8Array.from(atob(b64),c=>c.charCodeAt(0));
  const blob=new Blob([bytes],{type:"video/"+fmt});
  const url=URL.createObjectURL(blob);
  const vid=document.getElementById("avatarVideo");
  vid.src=url;
  vid.classList.add("active");
  S.speaking=true;
  document.getElementById("faceCard").classList.remove("thinking");
  document.getElementById("faceCard").classList.add("speaking");
  document.getElementById("discussBtn").textContent="Speaking...";
  setStatus("Speaking...","blue");
  vid.onended=()=>{
    vid.classList.remove("active");
    vid.src="";
    URL.revokeObjectURL(url);
    S.speaking=false;
    afterResponse();
  };
  vid.play().catch(()=>{
    vid.classList.remove("active");
    S.speaking=false;
    afterResponse();
  });
}

function playAudio(b64,fmt){
  try{S.recognition&&S.recognition.abort();}catch(e){}
  const bytes=Uint8Array.from(atob(b64),c=>c.charCodeAt(0));
  const blob=new Blob([bytes],{type:"audio/"+fmt});
  const url=URL.createObjectURL(blob);
  const audio=new Audio(url);
  if(!S.audioCtx) S.audioCtx=new AudioContext();
  if(S.audioCtx.state==='suspended') S.audioCtx.resume();
  try{
    const src=S.audioCtx.createMediaElementSource(audio);
    const analyser=S.audioCtx.createAnalyser();
    src.connect(analyser);analyser.connect(S.audioCtx.destination);
  }catch(e){}
  S.currentAudio=audio;
  S.speaking=true;
  document.getElementById("faceCard").classList.remove("thinking");
  document.getElementById("faceCard").classList.add("speaking");
  document.getElementById("discussBtn").textContent="Speaking...";
  setStatus("Speaking...","blue");
  audio.onended=()=>{
    if(S.currentAudio!==audio) return;
    S.currentAudio=null;S.speaking=false;afterResponse();
  };
  audio.play().catch(()=>{
    if(S.currentAudio!==audio) return;
    S.currentAudio=null;S.speaking=false;afterResponse();
  });
}

function afterResponse(){
  document.getElementById("faceCard").classList.remove("speaking");
  document.getElementById("faceCard").classList.remove("thinking");
  if(S.hideTimer) clearTimeout(S.hideTimer);
  S.hideTimer=setTimeout(()=>{
    document.getElementById("responseBubble").style.display="none";
    document.getElementById("transcriptBubble").style.display="none";
    S.hideTimer=null;
  },4000);
  if(S.continuous){
    setStatus("Listening...","gold");
    document.getElementById("discussBtn").textContent="Stop";
    setTimeout(()=>startListening(),1200);
  } else {
    document.getElementById("discussBtn").textContent="Discuss";
    document.getElementById("discussBtn").disabled=false;
    setStatus("Ready - Tap Discuss","green");
  }
}

function initRecognition(){
  const SR=window.SpeechRecognition||window.webkitSpeechRecognition;
  if(!SR) return false;
  S.recognition=new SR();
  S.recognition.lang=S.lang;
  S.recognition.interimResults=false;
  S.recognition.onresult=e=>{
    const text=e.results[0][0].transcript;
    const t=document.getElementById("transcriptBubble");
    t.textContent="You: "+text;t.style.display="block";
    sendText(text);
  };
  S.recognition.onerror=e=>{
    if(e.error!=="no-speech"&&e.error!=="aborted") setStatus("Mic error: "+e.error,"red");
    if(S.continuous&&!S.speaking&&e.error!=="aborted") setTimeout(()=>startListening(),400);
  };
  S.recognition.onend=()=>{
    if(S.continuous&&!S.speaking) setTimeout(()=>startListening(),400);
  };
  return true;
}

function startListening(){
  if(!S.ready||S.speaking) return;
  S.recognition=null;
  if(!initRecognition()){alert("Speech not supported - use chips below");return;}
  try{S.recognition.start();}catch(e){console.warn("recognition start:",e);}
}

function startDiscuss(){
  if(!S.ready){connectServer();return;}
  if(S.continuous){
    S.continuous=false;
    try{S.recognition&&S.recognition.stop();}catch(e){}
    document.getElementById("discussBtn").textContent="Discuss";
    setStatus("Ready - Tap Discuss","green");
    return;
  }
  S.continuous=true;
  document.getElementById("discussBtn").textContent="Stop";
  setStatus("Listening...","gold");
  startListening();
}

function sendText(text){
  if(!S.ws||!S.ready) return;
  try{S.recognition&&S.recognition.abort();}catch(e){}
  if(S.currentAudio){
    try{S.currentAudio.pause();S.currentAudio.src='';}catch(e){}
    S.currentAudio=null;
  }
  // Stop any playing video
  const vid=document.getElementById("avatarVideo");
  if(vid){vid.pause();vid.src="";vid.classList.remove("active");}
  document.getElementById("faceCard").classList.remove("speaking");
  if(S.hideTimer){clearTimeout(S.hideTimer);S.hideTimer=null;}
  S.sentCount++;
  S.speaking=true;
  document.getElementById("discussBtn").textContent="Thinking...";
  setStatus("Thinking...","gold");
  showThinking();
  S.ws.send(JSON.stringify({type:"speak",text,lang:S.lang==="hi-IN"?"hi":"en"}));
}

function toggleLang(){
  S.lang = S.lang==="en-IN" ? "hi-IN" : "en-IN";
  const btn=document.getElementById("langBtn");
  btn.textContent = S.lang==="hi-IN" ? "HI" : "EN";
  if(S.recognition){try{S.recognition.stop();}catch(e){} S.recognition=null;}
}

window.onload=()=>{
  const h=window.location.host;
  const proto=window.location.protocol==='https:'?'wss':'ws';
  document.getElementById('serverUrl').value=proto+'://'+h+'/ws/talk';
  setTimeout(connectServer,500);
};
</script>
</body>
</html>
'''
with open('/content/static/index.html', 'w') as f:
    f.write(index_html)
print("index.html written")

# ── Write knowledge_base.txt ──
print("Writing knowledge_base.txt...")
kb_text = """=== STABLE MONEY — OFFICIAL KNOWLEDGE BASE ===
Last updated: March 2026

COMPANY OVERVIEW
- Stable Money is India's leading fixed-income investment platform
- Headquartered in Bangalore, India
- Regulated investments only — all partner institutions are RBI-regulated

PRODUCTS
1. Fixed Deposits (FDs)
   - Partner institutions: 25+ NBFCs (Non-Banking Financial Companies) and Small Finance Banks
   - Returns: 8% to 9.5% per annum (vs 6.5-7% from regular banks)
   - Tenures: 6 months to 5 years
   - Minimum investment: Rs 1,000
   - DICGC-insured up to Rs 5 lakhs per depositor per bank
   - Suitable for: conservative investors seeking better-than-bank returns

2. Sukoon
   - No lock-in period — withdraw anytime
   - Returns: 7% to 8% per annum
   - Ideal for: idle/emergency money, short-term parking
   - Minimum investment: Rs 1,000

SAFETY & REGULATION
- All partner NBFCs and SFBs are RBI-regulated
- DICGC (Deposit Insurance and Credit Guarantee Corporation) insures deposits up to Rs 5 lakhs
- Stable Money is not a bank — it is a marketplace/platform
- Investments are held directly with partner institutions, not with Stable Money itself

COMPARISON WITH BANK FDs
- Regular bank FDs: ~6.5% to 7% returns
- Stable Money FDs: 8% to 9.5% returns
- Same DICGC insurance protection
- Wider choice of institutions

WHO SHOULD INVEST
- Individuals seeking safe, predictable returns
- Anyone with idle savings earning less than 7%
- Retirees or conservative investors
- People building an emergency fund (Sukoon product)

HOW TO GET STARTED
- Visit stablemoney.in
- Complete KYC (PAN + Aadhaar)
- Choose FD or Sukoon
- Invest from Rs 1,000

IMPORTANT DISCLAIMERS
- Past returns are indicative, not guaranteed
- Investments above Rs 5 lakhs are not DICGC-insured beyond the limit
- This is not a savings account — it is a term deposit
"""
with open('/content/knowledge_base.txt', 'w') as f:
    f.write(kb_text)
print("knowledge_base.txt written")

# ── Configure ngrok ──
from pyngrok import conf, ngrok
NGROK_TOKEN = "3BO5O8HXbvLr8j8TjHYPj5UFm4r_3BYg4BqamLqx7t7et34y9"
conf.get_default().auth_token = NGROK_TOKEN

# ── Start server ──
import nest_asyncio, uvicorn
nest_asyncio.apply()
os.chdir('/content')

ngrok.kill()
time.sleep(2)
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print('=' * 60)
print(f'PUBLIC URL: {public_url}')
print(f'Open this URL in your browser to use StableAI')
print('=' * 60)

config = uvicorn.Config('server:app', host='0.0.0.0', port=8000, log_level='info')
server_inst = uvicorn.Server(config)
await server_inst.serve()